# Chapter 3 - Lab 9: PydanticAI agent

**Goal**: build the agent with **PydanticAI**, which is schema-first: the final answer must validate against a typed model.

This is the same reference task used across every framework lab in Chapter 3:
**compare the Zenith X1 smartphone (`ZX1`) and the FrostLine refrigerator (`FL-340`) on contact rate**
(support tickets per 100 delivered orders) and write a short memo for the Head of Customer Care.
Comparing all frameworks on the *same* task makes differences in API style, abstractions and ergonomics easy to spot.

*Finance edition*: compare Apple and JPMorgan on P/E. Contact rate plays the role of P/E: a ratio
that lets a high-volume and a low-volume product be compared fairly (ZX1 = 15.0, FL-340 = 36.0).

**Tech stack**:
* **PydanticAI** (`pydantic-ai`).
* **OpenAI** `gpt-4o-mini`.

## 1. Install packages

In [ ]:
%pip install -q 'pydantic-ai-slim[openai]' pydantic python-dotenv

## 2. Set up the API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared task setup

The task, tools, dataset and prompts live in `common.py` (next to this notebook in the repository).

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%203/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    get_support_data,
    compute_contact_rate,
    support_data,
    system_message,
    input_message,
)

print("Tools loaded. Reference task:")
print(" ", input_message)

## 4. A typed output schema

In [ ]:
from pydantic import BaseModel, Field

class ContactRateMemo(BaseModel):
    zx1_contact_rate: float = Field(description="Tickets per 100 orders for ZX1")
    fl340_contact_rate: float = Field(description="Tickets per 100 orders for FL-340")
    higher_load_product: str = Field(description="Product line with the higher contact rate")
    memo: str = Field(description="Short memo for the Head of Customer Care")

## 5. Agent and tools

In [ ]:
from pydantic_ai import Agent, RunContext

agent = Agent("openai:gpt-4o-mini", system_prompt=system_message, output_type=ContactRateMemo)

@agent.tool
def support_data_tool(ctx: RunContext, product_line: str) -> dict:
    """Get the quarter's support tickets and delivered orders for a product line (e.g. ZX1, FL-340)."""
    return get_support_data(product_line).model_dump()

@agent.tool
def contact_rate_tool(ctx: RunContext, tickets: int, orders: int) -> float:
    """Compute the contact rate: support tickets per 100 orders."""
    return compute_contact_rate(tickets, orders)

## 6. Run

In [ ]:
result = await agent.run(input_message)
memo = result.output            # a validated ContactRateMemo object, not free text
print(memo.model_dump_json(indent=2))

## 7. Results
**What to notice about PydanticAI**: `result.output` is a typed object. A CRM or BI tool can store
`zx1_contact_rate` directly, with no parsing of free text. If the model returns a malformed answer,
PydanticAI re-prompts it automatically.